# Challenge 5 — Merge / Join

**Nombre:** Miriam Ivonne Castañeda  
**Clase:** Merge / Join  
**Dataset principal:** `dataset_limpio.csv`

## Objetivo

Integrar información de varias tablas y comparar diferentes tipos de `join`.

Para hacerlo de una forma sencilla, voy a separar el dataset limpio en tres tablas relacionadas:

- **Ventas:** información de cada venta.
- **Productos:** información del producto.
- **Vendedores:** información del vendedor y su región.

La idea es construir una tabla final que permita analizar las ventas.


## 1. Cargar el dataset limpio

In [ ]:
import pandas as pd

df = pd.read_csv("dataset_limpio.csv")

df.head()


## 2. Crear las tres fuentes

In [ ]:
# Tabla de ventas
ventas = df[[
    "fecha", "region_id", "vendedor", "producto_id",
    "cantidad", "precio_unitario", "total"
]].copy()

# Tabla de productos
productos = df[
    ["producto_id", "nombre_producto", "categoria"]
].drop_duplicates("producto_id").copy()

# Tabla de vendedores
vendedores = df[
    ["vendedor", "region_id"]
].drop_duplicates("vendedor").copy()

print("Ventas:", ventas.shape)
print("Productos:", productos.shape)
print("Vendedores:", vendedores.shape)


## 3. Identificar las llaves

- En `productos`, la llave es **producto_id**.
- En `ventas`, `producto_id` funciona como llave foránea.
- En `vendedores`, la llave es **vendedor**.
- En `ventas`, `vendedor` funciona como llave foránea.

La relación principal es de **uno a muchos**, porque un producto puede aparecer en muchas ventas y un vendedor puede realizar muchas ventas.


In [ ]:
print("Productos únicos:", productos["producto_id"].nunique())
print("Vendedores únicos:", vendedores["vendedor"].nunique())

print("\n¿Hay productos repetidos en su tabla?")
print(productos["producto_id"].duplicated().sum())

print("\n¿Hay vendedores repetidos en su tabla?")
print(vendedores["vendedor"].duplicated().sum())


## 4. Inner Join
Primero unimos ventas con productos.

In [ ]:
inner_productos = ventas.merge(
    productos,
    on="producto_id",
    how="inner"
)

print("Filas antes:", len(ventas))
print("Filas después del inner join:", len(inner_productos))

inner_productos.head()


### ¿Qué pasó?

El `inner join` conserva únicamente las filas que tienen coincidencia entre las dos tablas. En este caso, todas las ventas tienen un producto correspondiente, por lo que el número de filas se mantiene.


## 5. Left Join
Ahora hacemos un `left join` para conservar todas las ventas aunque alguna no tuviera producto correspondiente.

In [ ]:
left_productos = ventas.merge(
    productos,
    on="producto_id",
    how="left"
)

print("Filas después del left join:", len(left_productos))
print("\nValores faltantes generados:")
print(left_productos[["nombre_producto", "categoria"]].isna().sum())


### Comparación

En este caso, `inner` y `left` producen el mismo número de filas porque no hay ventas sin producto correspondiente.

El `left join` es útil porque asegura que la tabla principal de ventas se conserve completa.


## 6. Agregar la información de vendedores

In [ ]:
tabla_final = left_productos.merge(
    vendedores,
    on=["vendedor", "region_id"],
    how="left"
)

print("Filas de la tabla final:", len(tabla_final))
print("Columnas de la tabla final:", len(tabla_final.columns))

tabla_final.head()


## 7. Validación antes y después del merge

In [ ]:
print("VENTAS")
print("Filas:", len(ventas))
print("Columnas:", len(ventas.columns))
print("Vendedores únicos:", ventas["vendedor"].nunique())
print("Productos únicos:", ventas["producto_id"].nunique())

print("\nTABLA FINAL")
print("Filas:", len(tabla_final))
print("Columnas:", len(tabla_final.columns))
print("Vendedores únicos:", tabla_final["vendedor"].nunique())
print("Productos únicos:", tabla_final["producto_id"].nunique())

print("\nFaltantes en la tabla final:")
print(tabla_final.isna().sum())


## 8. Preguntas que podemos responder

In [ ]:
# 1. ¿Qué vendedor genera más ingresos?
ventas_por_vendedor = (
    tabla_final.groupby("vendedor")["total"]
    .sum()
    .sort_values(ascending=False)
)

print("Ventas por vendedor:")
display(ventas_por_vendedor.head())

# 2. ¿Qué producto genera más ingresos?
ventas_por_producto = (
    tabla_final.groupby(["producto_id", "nombre_producto"])["total"]
    .sum()
    .sort_values(ascending=False)
)

print("\nVentas por producto:")
display(ventas_por_producto.head())

# 3. ¿Qué categoría genera más ingresos?
ventas_por_categoria = (
    tabla_final.groupby("categoria")["total"]
    .sum()
    .sort_values(ascending=False)
)

print("\nVentas por categoría:")
display(ventas_por_categoria)


## 9. Conclusiones

1. El `inner join` y el `left join` conservaron el mismo número de filas porque las ventas tienen correspondencia con los productos.
2. El `left join` permite mantener todas las ventas aunque existiera alguna falta de correspondencia.
3. La tabla final integra información de ventas, productos y vendedores.
4. Las llaves principales utilizadas fueron `producto_id` y `vendedor`.
5. Con la tabla integrada se pueden obtener ventas por vendedor, producto y categoría.
